## Tools-create_tools 创建工具
使用 @tools 装饰器，
函数的文档字符串会成为工具的描述，帮助模型理解何时使用该工具。

类型提示是必需的，因为它们定义了工具的输入模式。文档字符串应简洁明了，以便模型理解工具的用途

In [2]:
from langchain.tools import tool


@tool
def search_database(query: str, limit: int = 10) -> str:
    """Search the customer database for records matching the query.

    Args:
        query: Search terms to look for
        limit: Maximum number of results to return
    """
    return f"Found {limit} results for '{query}'"

e:\code\LangChainDemo\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


### 自定义工具属性
#### 工具名称
默认情况下，工具名称来源于函数名称。如果需要更具描述性的名称，请对其进行覆盖：

In [3]:
@tool("web_search")  # Custom name
def search(query: str) -> str:
    """Search the web for information."""
    return f"Results for: {query}"

print(search.name)  # web_search

web_search


#### 工具描述
覆盖自动生成的工具描述，以便获得更清晰的模型指导

In [ ]:
@tool(
    "calculator",
    description="Performs arithmetic calculations. Use this for any math problems.",
)
def calc(expression: str) -> str:
    """Evaluate mathematical expressions."""
    return str(eval(expression))

### 高级模式定义
#### Pydantic


In [4]:
from pydantic import BaseModel, Field
from typing import Literal


class WeatherInput(BaseModel):
    """Input for weather queries"""

    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius", description="Temperature unit preference"
    ) # Literal 限制变量的取值范围
    include_forecast: bool = Field(default=False, description="Include 5-day forecast")


@tool(args_schema=WeatherInput)
def get_weather(
    location: str, units: str = "celsius", include_forecast: bool = False
) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

### 保留参数名称
以下参数名称为保留名称，不能用作工具参数。使用这些名称会导致运行时错误。
参数名称	  目的
config	    保留供RunnableConfig内部工具使用
runtime	    保留用于ToolRuntime参数（访问状态、上下文、存储）